# box walkthrough

A tour of `box` using the folder-per-experiment pattern: one experiment folder per run, params co-located with outputs, everything `ls`-inspectable.

In [1]:
import shutil
import pandas as pd
import box

shutil.rmtree('./catalog', ignore_errors=True)

## Set up a project

In [2]:
proj = box.init('walker', datastore='./catalog')

## Cache a shared preprocessed input

`@proj.compute_or_load` is a joblib-style cache scoped to the project. First call computes and saves; every subsequent call loads from disk.

In [3]:
@proj.compute_or_load('processed_input')
def preprocess():
    print('running expensive preprocessing...')
    return pd.DataFrame({'x': range(100)})

data = preprocess()
data.head()

running expensive preprocessing...


,x
0,0
1,1
2,2
3,3
4,4


In [4]:
# Second call: no print output -- loaded from disk
data = preprocess()
data.head()

,x
0,0
1,1
2,2
3,3
4,4


## Run a grid of experiments

Each grid point is its own experiment folder. Experiment identity is `(project, name, params)` -- same name with different params gives different folders.

In [5]:
def simulate(lr, prior):
    return pd.DataFrame({
        'step': range(10),
        'loss': [1.0 / (1 + i * lr) for i in range(10)],
    })

for lr in [0.01, 0.02, 0.05]:
    exp = proj.experiment('baseline', lr=lr, prior='uniform')
    shared = preprocess()  # cross-experiment cache; auto-tracked as an input
    exp.save(simulate(exp.lr, exp.prior), 'result')

## Inspect the layout on disk

Every experiment folder is self-contained: `params.json`, an experiment-level `manifest.json`, and one folder per artifact.

In [6]:
import subprocess
print(subprocess.check_output(['find', './catalog/walker', '-maxdepth', '3'], text=True))

./catalog/walker
./catalog/walker/2026-09-01__baseline__14a46f63
./catalog/walker/2026-09-01__baseline__14a46f63/result
./catalog/walker/2026-09-01__baseline__14a46f63/result/v1.parquet
./catalog/walker/2026-09-01__baseline__14a46f63/result/v1.manifest.json
./catalog/walker/2026-09-01__baseline__14a46f63/params.json
./catalog/walker/2026-09-01__baseline__14a46f63/manifest.json
./catalog/walker/2026-09-01__baseline__5f02a58a
./catalog/walker/2026-09-01__baseline__5f02a58a/result
./catalog/walker/2026-09-01__baseline__5f02a58a/result/v1.parquet
./catalog/walker/2026-09-01__baseline__5f02a58a/result/v1.manifest.json
./catalog/walker/2026-09-01__baseline__5f02a58a/params.json
./catalog/walker/2026-09-01__baseline__5f02a58a/manifest.json
./catalog/walker/2026-09-01__baseline__27f81a3c
./catalog/walker/2026-09-01__baseline__27f81a3c/result
./catalog/walker/2026-09-01__baseline__27f81a3c/result/v1.parquet
./catalog/walker/2026-09-01__baseline__27f81a3c/result/v1.manifest.json
./catalog/walker

## Meta-analyze across runs

In [7]:
runs = proj.runs()
runs.frame()

,name,folder,lr,prior
0,baseline,walker/2026-09-01__baseline__14a46f63,0.05,uniform
1,baseline,walker/2026-09-01__baseline__27f81a3c,0.02,uniform
2,baseline,walker/2026-09-01__baseline__5f02a58a,0.01,uniform


In [8]:
# Summary DataFrame -- one row per run
runs.where(prior='uniform').summarize(
    final_loss=lambda r: r.load('result')['loss'].iloc[-1],
    n_steps=lambda r: len(r.load('result')),
)

,name,folder,lr,prior,final_loss,n_steps
0,baseline,walker/2026-09-01__baseline__14a46f63,0.05,uniform,0.689655,10
1,baseline,walker/2026-09-01__baseline__27f81a3c,0.02,uniform,0.847458,10
2,baseline,walker/2026-09-01__baseline__5f02a58a,0.01,uniform,0.917431,10


In [9]:
# load_all yields (Run, artifact) pairs -- safe with the same-name grid pattern
for run, result in runs.load_all('result'):
    print(f"lr={run.params['lr']:.3f}   final_loss={result['loss'].iloc[-1]:.4f}")

lr=0.050   final_loss=0.6897
lr=0.020   final_loss=0.8475
lr=0.010   final_loss=0.9174


## Reproduce an older version

If we re-save `result` with different data, the previous version stays on disk. Load it by number.

In [10]:
exp = proj.experiment('baseline', lr=0.01, prior='uniform')  # reopens existing folder
exp.save(pd.DataFrame({'step': range(10), 'loss': [0.0] * 10}), 'result')  # v2

print('latest:', exp.load('result')['loss'].tolist())
print('v1    :', exp.load('result', version=1)['loss'].tolist())

latest: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
v1    : [1.0, 0.9900990099009901, 0.9803921568627451, 0.970873786407767, 0.9615384615384615, 0.9523809523809523, 0.9433962264150942, 0.9345794392523364, 0.9259259259259258, 0.9174311926605504]
